# M2 adım adım: sohbet içinde makaleyi bulan LangGraph agent'ı

M1'de her soru makalesiyle birlikte geliyordu. M2'de kullanıcı makaleyi söylemeden, bir sohbette art arda soru soruyor ve arada makale değiştiriyor. Agent'ın işi tek cümle:

> **soruyu geçmişle tamamla → makaleyi bul (gerekirse sor) → o makalede ara → kaynak göstererek cevapla**

Bu defter akışı gerçek kodla, node node gösterir:

0. Kurulum → 1. Akış diyagramı → 2. Sohbet verisi → 3. Yönlendirme kuralı (oylama) → 4. Bir sohbeti node node çalıştırmak → 5. "Hangisi?" (interrupt / resume) → 6. Takip sorusu (condense) → 7. Sonuçlar → 8. MLflow trace'leri → 9. Ne ayarlanabilir?

**Gerekenler:** GPU (bge-m3) ve Ollama (`ministral-3:14b`). 4-6. adımlar gerçek LLM çağrısı yapar. Büyük harfli değişkenler (`CONVERSATION_ID`, `QUESTION`…) senin değiştirmen için.

## 0 · Kurulum

`m2_agent.yaml`, `m1b_best.yaml`'ı `extends:` ile miras alır: retrieval ve cevap üretimi M1b'nin en iyi ayarlarıyla aynı. Yeni olan iki şey var: `eval.system` (neyin değerlendirileceği) ve `agent:` bölümü.

In [1]:
import json
import os
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)

from paperlens.config.loader import load_config

best_cfg = load_config("experiments/configs/m1b_best.yaml")
agent_cfg = load_config("experiments/configs/m2_agent.yaml")

print("eval.system:", best_cfg.eval.system, "->", agent_cfg.eval.system)
print("agent ayarları:")
print(json.dumps(agent_cfg.agent.model_dump(mode="json"), indent=2))

eval.system: single_paper -> agent
agent ayarları:
{
  "condense": {
    "enabled": true,
    "prompt_template": "prompts/condense.md",
    "triggers": [
      "\\bit\\b",
      "\\bits\\b",
      "\\bthem\\b",
      "\\bthat\\b",
      "\\bthose\\b",
      "\\bthere\\b",
      "\\bsame\\b",
      "\\b(former|latter)\\b",
      "^\\s*(and|what about|how about)\\b"
    ],
    "history_turns": 3
  },
  "route": {
    "top_k": 10,
    "min_lead": 0.3,
    "title_match_min": 90.0
  },
  "clarify": {
    "max_candidates": 3
  },
  "tracing": true
}


## 1 · Akış diyagramı

Graf `src/paperlens/graph/build.py` içinde kuruluyor; aşağıdaki diyagram koddan **otomatik** üretiliyor (elle çizilmedi). Node'lar:

| Node | Ne yapar | LLM? |
|---|---|---|
| `condense` | Takip sorusunu geçmişle tek başına anlaşılır hale getirir ("peki ne kadar büyük?" → "TweetQA veri seti ne kadar büyük?"). Geçmiş yoksa soruyu aynen geçirir. | evet (geçmiş varsa) |
| `resolve_paper` | Hangi makale? Soruda başlık geçiyor mu → yoksa 281 makalede arayıp chunk'ları makaleye göre oylar → karar: `named` / `vote` / `stay` / `clarify` | hayır |
| `clarify` | "Hangisini kastediyorsunuz?" diye sorar ve **durur** (`interrupt`). Kullanıcı cevap verince devam eder. | hayır |
| `retrieve` | Seçilen makalede arar (M1b retriever'ı, `paper_id` filtresi) | hayır |
| `generate` | Paragraf atıflı cevap (M1b prompt'u) | evet |
| `refuse` | Kullanıcının cevabı hiçbir makaleye uymadı → ret | hayır |

Kesikli oklar koşullu kenarlar: hangi yola gidileceğine bir önceki node'un çıktısı karar verir. Döngü yok; her tur mutlaka biter.

In [2]:
from paperlens.graph.build import build_agent_deps, build_graph, make_checkpointer
from paperlens.index.embedder import build_embedder
from paperlens.index.store import build_or_load
from paperlens.ingestion.sources import load_dataset
from paperlens.llm.client import OpenAICompatClient

loaded = load_dataset(agent_cfg.data)
papers = loaded.papers
titles = {p.paper_id: p.title for p in papers}
questions = {q.question_id: q for q in loaded.questions}
bundle = build_or_load(agent_cfg, papers, build_embedder(agent_cfg.embedding))
llm = OpenAICompatClient(agent_cfg.llm)
deps = build_agent_deps(agent_cfg, bundle, llm, titles)
graph = build_graph(deps, make_checkpointer())
print(f"{len(papers)} makale, {len(bundle.chunks)} chunk (corpus modu: hepsi tek index'te)\n")
print(graph.get_graph().draw_mermaid())

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

2026-10-21 23:32:32 [info     ] index cache hit                chunks=8360 key=0fdafd6f7ccb


281 makale, 8360 chunk (corpus modu: hepsi tek index'te)

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	condense(condense)
	resolve_paper(resolve_paper)
	clarify(clarify)
	retrieve(retrieve)
	generate(generate)
	refuse(refuse)
	__end__([<p>__end__</p>]):::last
	__start__ --> condense;
	clarify -.-> refuse;
	clarify -.-> retrieve;
	condense --> resolve_paper;
	resolve_paper -.-> clarify;
	resolve_paper -.-> retrieve;
	retrieve --> generate;
	generate --> __end__;
	refuse --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



Yukarıdaki metni https://mermaid.live adresine yapıştırırsan çizimi görürsün (README'de aynısı çizili hâlde var).

## 2 · Sohbet verisi

Değerlendirme için M1b'nin **aynı 250 sorusu** sentetik sohbetlere dağıtıldı (`src/paperlens/evals/conversations.py`, sabit seed): her sohbette 3 makale, sorular makale blokları hâlinde, bazen eski makaleye dönüş. Makale değişen turların yarısında kullanıcı makaleyi adıyla anıyor (3 farklı şablonla), yarısında anmıyor. Her turun bir tipi var:

- `first`: sohbetin ilk sorusu · `same`: aynı makaleden devam · `switch_signaled`: makale değişti ve adı geçiyor · `switch_unsignaled`: makale değişti, adı geçmiyor

In [3]:
from collections import Counter

from paperlens.evals.conversations import load_conversations

conversations = load_conversations(agent_cfg.conversations.path)
turns = [t for c in conversations for t in c]
print(f"{len(conversations)} sohbet, {len(turns)} tur")
print("tur tipleri:", dict(Counter(t.turn_type for t in turns)))
print(
    "şablonlar (sinyalli turlar):",
    dict(Counter(t.template for t in turns if t.template is not None)),
)

CONVERSATION_ID = "c003"  # <- başka bir sohbete bak
conv = next(c for c in conversations if c[0].conversation_id == CONVERSATION_ID)
for t in conv:
    print(f"\n[{t.turn_index}] {t.turn_type:18s} gold={titles[t.paper_id][:60]}")
    print("    kullanıcı:", t.text)

57 sohbet, 250 tur
tur tipleri: {'first': 57, 'switch_signaled': 67, 'same': 54, 'switch_unsignaled': 72}
şablonlar (sinyalli turlar): {1: 29, 0: 41, 2: 29}

[0] first              gold=Mining Supervisor Evaluation and Peer Feedback in Performanc
    kullanıcı: In Mining Supervisor Evaluation and, what evaluation metrics were used for the summarization task?

[1] switch_signaled    gold=What Does My QA Model Know? Devising Controlled Probes using
    kullanıcı: In What Does My QA, is WordNet useful for taxonomic reasoning for this task?

[2] same               gold=What Does My QA Model Know? Devising Controlled Probes using
    kullanıcı: After how many hops does accuracy decrease?

[3] switch_signaled    gold=Improving Correlation with Human Judgments by Integrating Se
    kullanıcı: In Improving Correlation with Human, what is a second order co-ocurrence matrix?

[4] switch_signaled    gold=What Does My QA Model Know? Devising Controlled Probes using
    kullanıcı: About the What Do

## 3 · Yönlendirme kuralı: oylama

`resolve_paper` LLM kullanmaz; kural `src/paperlens/graph/routing.py` içinde ve tek bir eşiği var:

1. Soruda bir makalenin **tam başlığı** (bulanık eşleşme ≥ 90) ya da **kısa adı** (başlıktaki ":" öncesi, büyük/küçük harfe duyarlı, tam kelime) geçiyorsa → `named`.
2. Değilse soru 281 makalede aranır (filtresiz, ilk 10 chunk) ve chunk'lar makaleye göre oylanır. `lead = 1. makalenin payı − 2.'nin payı`. `lead ≥ min_lead (0.3)` ise → `vote` (kazanan).
3. Fark yetersizse: aktif makale varsa → `stay` (insan da aynı makaleden devam ettiğinizi varsayardı), yoksa → `clarify`.

Aşağıda aynı kuralı iki soruda elle çalıştırıyoruz: biri çok genel, biri çok spesifik.

In [4]:
from paperlens.graph.routing import decide, vote

QUESTIONS = [  # <- kendi sorularını dene
    "Which dataset do they use?",
    "How many tweets are in the Arabic offensive language dataset?",
]
for text in QUESTIONS:
    votes = vote(bundle.dense.search(text, agent_cfg.agent.route.top_k))
    decision, paper = decide(text, None, votes, titles, agent_cfg.agent.route)
    print(f"\nSoru: {text}")
    for pid, share in votes[:3]:
        print(f"   {share:4.0%}  {titles[pid][:70]}")
    lead = votes[0][1] - (votes[1][1] if len(votes) > 1 else 0)
    print(f"   lead={lead:.2f} -> karar: {decision}", f"({titles[paper][:50]})" if paper else "")


Soru: Which dataset do they use?
    10%  Knowledge Graph Representation with Jointly Structural and Textual Enc
    10%  Distant supervision for emotion detection using Facebook reactions
    10%  Adaptive Semi-supervised Learning for Cross-domain Sentiment Classific
   lead=0.00 -> karar: clarify 

Soru: How many tweets are in the Arabic offensive language dataset?
    80%  Arabic Offensive Language on Twitter: Analysis and Experiments
    10%  Predicting the Type and Target of Offensive Posts in Social Media
    10%  Offensive Language Detection: A Comparative Analysis
   lead=0.70 -> karar: vote (Arabic Offensive Language on Twitter: Analysis and)


## 4 · Bir sohbeti node node çalıştırmak

`graph.stream(..., stream_mode="updates")` her node bittiğinde o node'un **state'e yazdığı alanları** verir. Böylece agent'ın her adımda ne düşündüğünü görürüz. `thread_id` sohbetin kimliği: aynı `thread_id` ile gelen sonraki tur, checkpointer sayesinde `history` ve `active_paper_id`'yi hatırlar.

Bu hücre 2. adımdaki sohbeti gerçek LLM ile oynatır. Agent "hangisi?" diye sorarsa simüle kullanıcı doğru başlığı söyler (5. adımda bunu elle yapacağız).

In [5]:
from langgraph.types import Command


def show(update: dict) -> None:
    for node, out in update.items():
        if node == "__interrupt__":
            print(f"   ⏸  interrupt: {out[0].value['message'][:150]}")
            continue
        out = out or {}
        if node == "condense":
            print(f"   condense      -> standalone: {out.get('standalone')!r}")
        elif node == "resolve_paper":
            top = ", ".join(f"{titles[p][:28]}… {s:.0%}" for p, s in out["candidates"][:3])
            print(f"   resolve_paper -> {out['decision']} | oylar: {top}")
        elif node == "clarify":
            pid = out.get("resolved_paper_id")
            print(f"   clarify       -> cevap eşleşti: {titles.get(pid) if pid else None}")
        elif node == "retrieve":
            secs = [sc.chunk.section for sc in out["retrieved"]]
            print(f"   retrieve      -> {len(secs)} chunk, bölümler: {secs}")
        elif node in ("generate", "refuse"):
            a = out["answer"]
            print(f"   {node:13s} -> cevap: {a.text!r} | atıf: {a.evidence_ids}")


config = {"configurable": {"thread_id": f"nb-{CONVERSATION_ID}"}}
for t in conv:
    q = questions[t.question_id]
    print(f"\n[{t.turn_index}] {t.turn_type} | kullanıcı: {t.text}")
    stream = graph.stream({"question": q, "text": t.text}, config, stream_mode="updates")
    for update in stream:
        show(update)
    if graph.get_state(config).next:  # interrupt'ta durdu: simüle kullanıcı cevaplıyor
        print(f"   kullanıcı cevabı: {titles[t.paper_id]!r}")
        for update in graph.stream(
            Command(resume=titles[t.paper_id]), config, stream_mode="updates"
        ):
            show(update)
    state = graph.get_state(config).values
    ok = "✓" if state["active_paper_id"] == t.paper_id else "✗"
    print(
        f"   {ok} aktif makale: {titles.get(state['active_paper_id'] or '')} | gold cevap: {q.references[0].answer_text[:80]!r}"
    )


[0] first | kullanıcı: In Mining Supervisor Evaluation and, what evaluation metrics were used for the summarization task?
   condense      -> standalone: 'In Mining Supervisor Evaluation and, what evaluation metrics were used for the summarization task?'
   resolve_paper -> named | oylar: Revisiting Summarization Eva… 30%, Mining Supervisor Evaluation… 30%, Ranking Sentences for Extrac… 20%
   retrieve      -> 5 chunk, bölümler: ['Abstract', 'Conclusions and Further Work', 'Dataset', 'Evaluation of auto-generated summaries', 'Sentence Classification']


   generate      -> cevap: 'ROUGE unigram score' | atıf: ['1712.00991::s8p0']
   ✓ aktif makale: Mining Supervisor Evaluation and Peer Feedback in Performance Appraisals | gold cevap: 'ROUGE BIBREF22 unigram score'

[1] switch_signaled | kullanıcı: In What Does My QA, is WordNet useful for taxonomic reasoning for this task?
   condense      -> standalone: 'In What Does My QA, is WordNet useful for taxonomic reasoning for this task?'
   resolve_paper -> named | oylar: What Does My QA Model Know? … 90%, Multi-class Hierarchical Que… 10%
   retrieve      -> 5 chunk, bölümler: ['Introduction', 'Dataset Probes and Construction ::: WordNetQA', 'Dataset Probes and Construction ::: WordNetQA ::: Example Generation @!START@$\\textsc {gen}(\\tau )$@!END@.', 'Related Work', 'Abstract']


   generate      -> cevap: 'WordNet is used to measure model competence in taxonomic knowledge in different settings (including hypernymy, hyponymy, and synonymy detection)' | atıf: ['1912.13337::s0p5']
   ✓ aktif makale: What Does My QA Model Know? Devising Controlled Probes using Expert Knowledge | gold cevap: 'Unanswerable'

[2] same | kullanıcı: After how many hops does accuracy decrease?
   condense      -> standalone: 'After how many hops does accuracy decrease?'
   resolve_paper -> stay | oylar: What Does My QA Model Know? … 20%, A Human-AI Loop Approach for… 10%, Elephant in the Room: An Eva… 10%
   retrieve      -> 5 chunk, bölümler: ['Results and Findings ::: How well do pre-trained MCQA models do?', 'Introduction', 'Results and Findings ::: Can Models Be Effectively Inoculated?', 'Figures and Tables', 'Results and Findings ::: Are Models Consistent across Clusters?']


   generate      -> cevap: '1-hop links to 2-hops' | atıf: ['1912.13337::s0p8']
   ✓ aktif makale: What Does My QA Model Know? Devising Controlled Probes using Expert Knowledge | gold cevap: '1-hop links to 2-hops'

[3] switch_signaled | kullanıcı: In Improving Correlation with Human, what is a second order co-ocurrence matrix?
   condense      -> standalone: 'In Improving Correlation with Human, what is a second order co-ocurrence matrix?'
   resolve_paper -> named | oylar: Improving Correlation with H… 100%
   retrieve      -> 5 chunk, bölümler: ['Method', 'Abstract', 'Relatedness Measures', 'Conclusion and Future Work', 'Introduction']


   generate      -> cevap: 'a co–occurrence matrix must be constructed. Second, this matrix is used to construct a second–order co–occurrence vector for each concept in a pair of concepts to be measured for relatedness.' | atıf: ['1609.00559::s5p0']
   ✓ aktif makale: Improving Correlation with Human Judgments by Integrating Semantic Similarity with Second--Order Vectors | gold cevap: 'frequencies of the other words which occur with both of them (i.e., second order'

[4] switch_signaled | kullanıcı: About the What Does My QA paper: are the automatically constructed datasets subject to quality control?
   condense      -> standalone: 'About the What Does My QA paper: are the automatically constructed datasets subject to quality control?'
   resolve_paper -> named | oylar: What Does My QA Model Know? … 50%, ReviewQA: a relational aspec… 20%, A Joint Model for Multimodal… 10%
   retrieve      -> 5 chunk, bölümler: ['Discussion and Conclusion', 'Discussion and Conclusion', 'Introduction', 

   generate      -> cevap: 'automatically generate datasets [...] is much harder to validate the quality of such data at such a scale and such varying levels of complexity' | atıf: ['1912.13337::s20p4']
   ✓ aktif makale: What Does My QA Model Know? Devising Controlled Probes using Expert Knowledge | gold cevap: 'No'


## 5 · "Hangisi?": interrupt ve resume elle

Aktif makale yokken genel bir soru gelince `clarify` node'u `interrupt()` çağırır: graf **durur**, state checkpointer'da saklanır ve `invoke` aday başlıklarıyla bir mesaj döner. Kullanıcı cevap verince `Command(resume=<cevap>)` ile graf kaldığı yerden devam eder. Cevap `match_title` ile önce adaylarla, sonra tüm başlıklarla eşlenir; hiçbirine uymazsa `refuse`.

In [6]:
from paperlens.domain import Question

ASK = "Which datasets did they experiment with?"
REPLY = "External Lexical Information for Multilingual Part-of-Speech Tagging"  # <- "bilmiyorum" yazıp refuse'u gör

config = {"configurable": {"thread_id": "nb-clarify"}}
q = Question(question_id="demo", paper_id="", question=ASK, references=[])
state = graph.invoke({"question": q, "text": ASK}, config)
print("graf durdu mu?", "__interrupt__" in state)
print("agent:", state["__interrupt__"][0].value["message"])
print("bekleyen node:", graph.get_state(config).next)

state = graph.invoke(Command(resume=REPLY), config)
print("\nkullanıcı:", REPLY)
print("karar:", state["decision"], "| clarified:", state["clarified"])
print("cevap:", state["answer"].text, "| atıf:", state["answer"].evidence_ids)

graf durdu mu? True
agent: Which paper do you mean? 1) A Joint Model for Multimodal Document Quality Assessment 2) Adaptive Semi-supervised Learning for Cross-domain Sentiment Classification 3) STEP: Sequence-to-Sequence Transformer Pre-training for Document Summarization
bekleyen node: ('clarify',)



kullanıcı: External Lexical Information for Multilingual Part-of-Speech Tagging
karar: clarify | clarified: True
cevap: the multilingual dataset provided during the SPMRL parsing shared task, Universal Dependencies (v1.2) corpus set | atıf: ['1606.03676::s1p2', '1606.03676::s0p6']


## 6 · Takip sorusu: `condense`

QASPER soruları zaten tek başına anlaşılır, bu yüzden 250 soruluk değerlendirme `condense`'i ölçemez (raporda `condense.changed_rate` ile sadece "gereksiz yere değiştiriyor mu" kontrol ediliyor). Bunu göstermek için elle yazılmış takip sohbetleri var: `evals/conversations/followups_demo.jsonl` (10 sohbet). Tümünün dökümü: `make converse` → `docs/m2_demo.md`.

In [7]:
from paperlens.graph.converse import converse, load_demo

demo = load_demo(Path("evals/conversations/followups_demo.jsonl"))
DEMO_IDS = ["d01", "d04"]  # <- başka demo sohbetler: d01 ... d10
print(converse(graph, [c for c in demo if c[0].conversation_id in DEMO_IDS], titles))

# M2 follow-up demo

Generated by `paperlens converse`; one section per chat.

## d01

**1. User:** In TWEETQA, what is the size of this dataset?

- standalone question: In TWEETQA, what is the size of this dataset?
- corpus vote: TWEETQA: A Social Media Focused Question Answering Dataset (30%), A multi-layer approach to disinformation detection on Twitter (20%), Deep Learning for Detecting Cyberbullying Across Multiple Social Media Platforms (10%)
- decision: `named` -> TWEETQA: A Social Media Focused Question Answering Dataset
- **answer:** 10,898 articles, 17,794 tweets, and 13,757 crowdsourced question-answer pairs (cited paragraphs: 1907.06292::s2p10)

**2. User:** And which evaluation metrics do they use on it?

- standalone question: And which evaluation metrics do they use on it?
- corpus vote: Shallow reading with Deep Learning: Predicting popularity of online content using only its title (10%), Clickbait? Sensational Headline Generation with Auto-tuned Reinforcement Learning 

## 7 · Sonuçlar (250 soru, aynı sorular üç sistemde)

| Sistem | Nasıl çalışır |
|---|---|
| `m1b_best` | Doğru makale **verili** (M1 kurgusu): tavan |
| `m2_corpus_rag` | Aynı pipeline, ama her tur 281 makalede filtresiz aranır; geçmiş yok, yönlendirme yok: taban |
| `m2_agent` | LangGraph agent'ı, sohbet sırasıyla |

- **paper_acc**: cevabın dayandığı makale doğru mu (corpus_rag için en üstteki chunk'ın makalesi).
- **clarify**: agent'ın "hangisi?" diye sorduğu turların oranı. Simüle kullanıcı hep doğru başlığı verdiği için soru sormak puanı "bedavaya" artırır; bu yüzden `f1_no_clarify` (soru sorulmayan turlar) ayrıca var.
- **wrong_paper**: sormadan, ret etmeden yanlış makaleden verilen cevaplar.

In [8]:
from paperlens.evals.report import load_run
from paperlens.evals.stats import paired_bootstrap

RESULTS = Path("evals/results")
runs = {name: load_run(RESULTS / name) for name in ("m1b_best", "m2_corpus_rag", "m2_agent")}
COLS = {
    "answer_f1": "answer_f1",
    "evidence_f1": "ev_f1",
    "judge.accuracy": "judge",
    "route.paper_accuracy": "paper_acc",
    "route.clarify_rate": "clarify",
    "route.wrong_paper_rate": "wrong_paper",
    "answer_f1.no_clarify": "f1_no_clarify",
    "latency_p95_ms": "p95_ms",
}
print(f"{'run':15s}" + "".join(f"{c:>14s}" for c in COLS.values()))
for name, run in runs.items():
    row = [run.metrics.get(k) for k in COLS]
    print(f"{name:15s}" + "".join(f"{'-':>14s}" if v is None else f"{v:14.3f}" for v in row))


def f1s(run, key="answer_f1"):
    return {r.question_id: getattr(r, key) for r in run.records}


print("\nFark (paired bootstrap, %95 CI):")
for a, b in [("m2_agent", "m2_corpus_rag"), ("m2_agent", "m1b_best")]:
    for key in ("answer_f1", "evidence_f1"):
        res = paired_bootstrap(f1s(runs[a], key), f1s(runs[b], key))
        print(f"  {a} - {b:14s} {key:12s} {res.mean_diff:+.3f} [{res.low:+.3f}, {res.high:+.3f}]")

run                 answer_f1         ev_f1         judge     paper_acc       clarify   wrong_paper f1_no_clarify        p95_ms
m1b_best                0.440         0.491         0.568             -             -             -             -      1642.370
m2_corpus_rag           0.216         0.181         0.248         0.456         0.000         0.492         0.216      1862.520
m2_agent                0.290         0.293         0.344         0.676         0.076         0.220         0.279      1749.370

Fark (paired bootstrap, %95 CI):
  m2_agent - m2_corpus_rag  answer_f1    +0.074 [+0.036, +0.111]
  m2_agent - m2_corpus_rag  evidence_f1  +0.112 [+0.068, +0.156]
  m2_agent - m1b_best       answer_f1    -0.151 [-0.199, -0.103]
  m2_agent - m1b_best       evidence_f1  -0.198 [-0.256, -0.141]


### Tur tipine göre

Agent'ın güçlü ve zayıf olduğu yerler burada görünür: `same` turlarında aktif makaleyi hatırladığı için corpus_rag'den çok daha iyi olmalı; `switch_unsignaled` turlarında ise tasarım gereği eski makalede kalır.

In [9]:
KINDS = ("first", "same", "switch_signaled", "switch_unsignaled")
print(
    f"{'tur tipi':20s}{'n':>5s}"
    + "".join(f"{n + ' acc':>18s}{n + ' f1':>17s}" for n in ("corpus", "agent"))
)
for kind in KINDS:
    n = int(runs["m2_agent"].metrics.get(f"n.turn.{kind}", 0))
    vals = []
    for name in ("m2_corpus_rag", "m2_agent"):
        m = runs[name].metrics
        vals += [m.get(f"route.paper_accuracy.{kind}", 0), m.get(f"answer_f1.turn.{kind}", 0)]
    print(
        f"{kind:20s}{n:5d}"
        + "".join(f"{v:18.3f}" if i % 2 == 0 else f"{v:17.3f}" for i, v in enumerate(vals))
    )

m = runs["m2_agent"].metrics
print("\nSinyalli geçiş yakalama, şablon başına:")
for i, template in enumerate(agent_cfg.conversations.signal_templates):
    print(f"  t{i} {template!r:45s} {m.get(f'route.signaled_catch.t{i}', float('nan')):.3f}")
print("agent kararları:", dict(Counter(r.decision for r in runs["m2_agent"].records)))

tur tipi                n        corpus acc        corpus f1         agent acc         agent f1
first                  57             0.596            0.200             0.982            0.396
same                   54             0.259            0.236             0.630            0.264
switch_signaled        67             0.687            0.304             1.000            0.391
switch_unsignaled      72             0.278            0.131             0.167            0.131

Sinyalli geçiş yakalama, şablon başına:
  t0 'In the paper "{title}", {question}'          1.000
  t1 'In {short_title}, {question}'                1.000
  t2 'About the {short_title} paper: {question}'   1.000
agent kararları: {'vote': 34, 'named': 100, 'stay': 97, 'clarify': 19}


## 8 · MLflow trace'leri

`agent.tracing: true` iken her sohbet bir **trace**, her node bir **span** olarak MLflow'a yazılır (`mlflow.langchain.autolog()`). Arayüzde görmek için:

```bash
uv run mlflow ui --backend-store-uri sqlite:///mlflow.db
```

→ http://localhost:5000 → `m2` deneyi → **Traces** sekmesi. Bir trace'e tıklayınca solda node ağacı (condense → resolve_paper → …), sağda her span'in girdisi/çıktısı görünür. Aynı bilgiye koddan da ulaşılabilir:

In [10]:
import mlflow

mlflow.set_tracking_uri(agent_cfg.eval.mlflow_tracking_uri)
experiment = mlflow.get_experiment_by_name("m2")
traces = mlflow.search_traces(
    locations=[experiment.experiment_id], return_type="list", max_results=200
)
print(f"{len(traces)} trace bulundu (m2 deneyindeki tüm agent koşuları)")
trace = traces[0]
names = [s.name for s in trace.data.spans]
print("bir trace'teki span'ler:", names[:20], "..." if len(names) > 20 else "")

171 trace bulundu (m2 deneyindeki tüm agent koşuları)
bir trace'teki span'ler: ['c056', 'LangGraph', 'condense', 'resolve_paper', '_after_resolve', 'clarify', 'LangGraph', 'clarify', '_after_clarify', 'retrieve', 'generate', 'LangGraph', 'condense', 'resolve_paper', '_after_resolve', 'retrieve', 'generate', 'LangGraph', 'condense', 'resolve_paper'] ...


## 9 · Ne ayarlanabilir?

| Ayar | Ne değişir |
|---|---|
| `agent.route.min_lead` (0.3) | Yükseltirsen: daha az yanlış makale, daha çok "hangisi?" ve daha az yakalanan sinyalsiz geçiş. Düşürürsen tersi. Raporda `route.clarify_rate` ile `route.wrong_paper_rate`'i birlikte izle. |
| `agent.route.top_k` (10) | Oylamaya giren chunk sayısı. Fazlası payları yumuşatır, genel sorular daha çok makaleye dağılır. |
| `agent.route.title_match_min` (90) | Başlık eşleşmesi ne kadar katı. |
| `agent.condense.enabled` | Kapatırsan takip soruları ("peki ne kadar büyük?") olduğu gibi aranır; `make converse` ile farkı gör. |
| `conversations.signal_prob` | Kullanıcının makaleyi ne sıklıkla andığı: 0 ve 1 ile koşup agent'ın isme ne kadar dayandığını ölç. |

Bir ayarı değiştirmek için `experiments/configs/m2_agent.yaml`'ı kopyala, `eval.run_name`'i değiştir ve `uv run paperlens eval -c <dosya>` çalıştır.

In [11]:
bundle.dense.close()  # Qdrant yerel kilidini bırak (başka bir süreç index'i açabilsin)